# EXP009_DETECTOR_V1_E01 — resume
Engineering-only NumPy-scalar JSON correction. **Stop the original notebook before running this one.** Keep the existing Drive results. Run All, mount Drive and upload the E01 bundle once. No GPU, model training or NASA TEST access.

Original source/configuration and cache identity remain unchanged. The E01 launcher records its own amendment provenance and extends only the JSON writer. Any environment/hash mismatch stops safely. This notebook has been syntax-checked locally; hosted execution has not yet been tested.


In [ ]:
from google.colab import drive,files
from pathlib import Path
import sys,json,hashlib,zipfile,shutil,subprocess,venv,datetime,os
assert (3,11)<=sys.version_info[:2]<=(3,13)
drive.mount('/content/drive')
BASE=Path('/content/drive/MyDrive/NASA_GCN/EXP009_DETECTOR_V1')
OUT=BASE/'results'
assert (OUT/'RUN_FREEZE.json').is_file(), 'Existing EXP009 results not found. Do not start a replacement run.'
E01ZIP=BASE/'EXP009_DETECTOR_V1_E01_bundle.zip'
EXPECTED='c42866d9056c129a2c2ca9bf6bbee3c1048ffbd85fdc1e1d85286b4b1564bd5d'
if not E01ZIP.exists():
    uploaded=files.upload()
    matches=[data for data in uploaded.values()if hashlib.sha256(data).hexdigest()==EXPECTED]
    assert len(matches)==1, 'Upload the supplied E01 bundle.'
    tmp=E01ZIP.with_suffix('.tmp');tmp.write_bytes(matches[0]);os.replace(tmp,E01ZIP)
assert hashlib.sha256(E01ZIP.read_bytes()).hexdigest()==EXPECTED
WORK=Path('/content/EXP009_DETECTOR_V1_E01_resume');WORK.mkdir(exist_ok=True)
def safe_extract(archive,destination):
    with zipfile.ZipFile(archive)as z:
        assert len(z.namelist())==len(set(z.namelist()))
        for name in z.namelist():assert not Path(name).is_absolute()and '..'not in Path(name).parts
        z.extractall(destination)
safe_extract(E01ZIP,WORK)
AMEND=WORK/'amendment';em=json.loads((AMEND/'E01_MANIFEST.json').read_text())
assert hashlib.sha256((WORK/'parent_bundle.zip').read_bytes()).hexdigest()==em['parent_bundle_sha256']
for name,h in em['files'].items():assert hashlib.sha256((AMEND/name).read_bytes()).hexdigest()==h
old_bundle=BASE/'EXP009_DETECTOR_V1_bundle.zip'
if old_bundle.exists():assert hashlib.sha256(old_bundle.read_bytes()).hexdigest()==em['parent_bundle_sha256']
PARENT=WORK/'parent';PARENT.mkdir(exist_ok=True);safe_extract(WORK/'parent_bundle.zip',PARENT)
EVIDENCE=BASE/'engineering_E01'
LOGS=BASE/'session_logs';LOGS.mkdir(exist_ok=True)
STAMP=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
print('Original bundle and separate E01 amendment verified.')


In [ ]:
# Reuse the existing numerical environment. Only provision it if the runtime was reset.
ENV=Path('/content/exp009_cpu_env');PY=ENV/'bin/python'
if not PY.exists():
    venv.EnvBuilder(with_pip=False).create(ENV)
    with (LOGS/f'{STAMP}_E01_setup.log').open('w')as log:
        result=subprocess.run([sys.executable,'-m','pip','--python',str(PY),'install','pip','numpy==2.3.5','pandas==2.2.3'],stdout=log,stderr=subprocess.STDOUT)
    assert result.returncode==0,(LOGS/f'{STAMP}_E01_setup.log').read_text()[-6000:]
CMD=[str(PY),'-u',str(AMEND/'resume_e01.py'),'--parent',str(PARENT),'--output',str(OUT),'--evidence',str(EVIDENCE)]
# Checks actual commits and original environment before any detector work.
subprocess.run(CMD+['--preflight-only'],check=True)
# Synthetic serializer/interruption/resume regression; no project detector selection.
subprocess.run([str(PY),str(AMEND/'test_e01.py'),str(PARENT)],check=True)
print('Resume preflight and engineering regression checks passed.')


In [ ]:
with (LOGS/f'{STAMP}_E01_run.log').open('w')as log:
    p=subprocess.Popen(CMD,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in p.stdout:print(line,end='');log.write(line);log.flush()
    rc=p.wait()
assert rc==0, 'Stopped safely. Preserve results and logs; rerun E01 only after resolving any reported integrity/environment issue.'
print((OUT/'STUDY_CLOSE.json').read_text())
print((EVIDENCE/'POST_E01_PRESERVATION.json').read_text())


In [ ]:
EXPORT=BASE/'exports';EXPORT.mkdir(exist_ok=True)
archive=EXPORT/f'EXP009_DETECTOR_V1_E01_results_{STAMP}.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED)as z:
    for folder,prefix in [(OUT,'results'),(EVIDENCE,'engineering_E01'),(LOGS,'session_logs')]:
        for p in sorted(folder.rglob('*')):
            if p.is_file()and p.name!='RUN.lock':z.write(p,prefix+'/'+str(p.relative_to(folder)))
    z.write(E01ZIP,'input/EXP009_DETECTOR_V1_E01_bundle.zip')
h=hashlib.sha256(archive.read_bytes()).hexdigest();archive.with_suffix('.sha256').write_text(h+'  '+archive.name+'\n')
print('Saved:',archive);print('SHA-256:',h)
print('STOP: submit the completed results and executed E01 notebook for audit. No final NASA evaluation is run here.')
files.download(str(archive))
